# WhisperColab

OpenAI **Whisper Large-v3** 음성 인식 모델을 Colab GPU에서 실행하고, **TryCloudflare**로 임시 웹패널을 공개합니다. 파일을 업로드하면 TXT 파일을 자동 생성합니다.

> **중요:** Colab 무료 티어는 GPU 할당을 보장하지 않으며, 웹 UI 사용 시 런타임이 조기 종료될 수 있습니다. 개인적인 단기 실험 용도로만 사용해 주세요.

1. 상단 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 선택
2. 메뉴 **런타임 → 모두 실행** (또는 아래 셀 순서대로 실행)
3. 출력에 표시되는 TryCloudflare 주소를 열고 사용자명 `whisper`, 출력된 비밀번호로 로그인
4. 음성 파일을 선택하면 자동 변환 → **TXT 다운로드**

**모델:** OpenAI Whisper Large-v3 (faster-whisper / CTranslate2, GPU INT8+FP16 연산). OpenAI API 키는 필요하지 않습니다.


In [ ]:
# 1. Colab의 [런타임 > 런타임 유형 변경 > T4 GPU]를 먼저 선택하세요.
import os
import subprocess
import sys
from pathlib import Path

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
)
if gpu.returncode != 0:
    raise RuntimeError("GPU가 없습니다. 런타임 > 런타임 유형 변경 > T4 GPU를 선택하고 다시 실행해 주세요.")
print("연결된 GPU:", gpu.stdout.strip())

repo_dir = Path("/content/WhisperColab")
if (repo_dir / ".git").exists():
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=True)
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/dariring/WhisperColab.git", str(repo_dir)],
        check=True,
    )

# CUDA 12 / cuDNN 9: faster-whisper(CTranslate2)의 GPU 필수 라이브러리
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(repo_dir / "requirements.txt"),
     "nvidia-cublas-cu12", "nvidia-cudnn-cu12==9.*"],
    check=True,
)

cloudflared = Path("/content/cloudflared")
if not cloudflared.is_file():
    from urllib.request import urlretrieve
    urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        str(cloudflared),
    )
    cloudflared.chmod(0o755)
print("설치 완료. 다음 셀을 실행하면 웹패널과 TryCloudflare 터널이 열립니다.")



In [ ]:
# 2. 웹패널을 열고, 아래 출력에 표시되는 비밀번호로 로그인하세요.
import html
import os
import re
import secrets
import subprocess
import sys
import time
from pathlib import Path
from urllib.request import urlopen
from IPython.display import HTML, display

# 같은 세션에서 다시 실행하면 이전에 띄운 프로세스를 정리합니다.
for key in ("whisper_tunnel_process", "whisper_server_process"):
    old = globals().get(key)
    if old is not None and old.poll() is None:
        old.terminate()
        try:
            old.wait(timeout=8)
        except subprocess.TimeoutExpired:
            old.kill()

import nvidia.cublas
import nvidia.cudnn
lib_dirs = [
    str(Path(nvidia.cublas.__path__[0]) / "lib"),
    str(Path(nvidia.cudnn.__path__[0]) / "lib"),
]
env = os.environ.copy()
env["LD_LIBRARY_PATH"] = os.pathsep.join(lib_dirs + [env.get("LD_LIBRARY_PATH", "")])
panel_password = secrets.token_urlsafe(18)
env["PANEL_PASSWORD"] = panel_password
env["HF_HUB_DISABLE_TELEMETRY"] = "1"

server_log_path = Path("/content/whispercolab-server.log")
with server_log_path.open("w") as log:
    whisper_server_process = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "app:app", "--host", "127.0.0.1", "--port", "7860"],
        cwd=str(repo_dir), env=env, stdout=log, stderr=subprocess.STDOUT,
    )

ready = False
for _ in range(60):
    if whisper_server_process.poll() is not None:
        break
    try:
        with urlopen("http://127.0.0.1:7860/api/health", timeout=2) as response:
            if response.status == 200:
                ready = True
                break
    except Exception:
        pass
    time.sleep(1)
if not ready:
    raise RuntimeError("웹서버 시작 실패:\n" + server_log_path.read_text()[-3000:])

tunnel_log_path = Path("/content/whispercolab-tunnel.log")
with tunnel_log_path.open("w") as log:
    whisper_tunnel_process = subprocess.Popen(
        [str(cloudflared), "tunnel", "--url", "http://127.0.0.1:7860", "--no-autoupdate"],
        stdout=log, stderr=subprocess.STDOUT,
    )

tunnel_url = None
for _ in range(90):
    logs = tunnel_log_path.read_text(errors="replace")
    found = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", logs)
    if found:
        tunnel_url = found.group(0)
        break
    if whisper_tunnel_process.poll() is not None:
        break
    time.sleep(1)
if tunnel_url is None:
    raise RuntimeError("TryCloudflare 터널 시작 실패:\n" + tunnel_log_path.read_text()[-3000:])

print("로그인 사용자명: whisper")
print("로그인 비밀번호:", panel_password)
print("웹패널 주소:", tunnel_url)
display(HTML(
    '<a href="' + html.escape(tunnel_url, quote=True) + '" target="_blank" '
    'style="display:inline-block;background:#202e43;color:#fff;padding:13px 20px;'
    'border-radius:10px;text-decoration:none;font-weight:bold;">Whisper 웹패널 열기 ↗</a>'
))
print("Colab 런타임이 중지되면 임시 주소와 웹패널도 함께 종료됩니다.")

